# Camada GOLD - Analise base de dados "Viagens a Serviço do Portal da Transparência do Governo Federal"

Este notebook le a camada **SILVER** (ja limpa e tipada) e responde 3 perguntas
de negocio. Para cada uma: a consulta SQL, a tabela com o resultado e um grafico.

**Perguntas a responder:**

1. Os 5 órgãos com maior custo total?
2. Qual o meio de transporte mais usado nos trechos?
3. Qual UF de destino aparece em mais trechos?


| Pergunta                                              | Tabela(s)                            | Tipo de gráfico    |                                                                 
| ----------------------------------------------------- | ------------------------------------ | ------------------ | 
| **Os 5 órgãos com maior custo total?**                | `silver_viagem` + `silver_pagamento` | Barras horizontais | 
| **Qual UF de destino aparece em mais trechos?**       | `silver_trecho`                      | Gráfico de Rosca   | 
| **Qual o meio de transporte mais usado nos trechos?** | `silver_trecho`                      | Barras             | 



## Conexao com o banco 


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

import banco

conexao = banco.conectar()

def consultar(sql):
    """Roda um SELECT e devolve o resultado como DataFrame do pandas."""
    return pd.read_sql(sql, conexao)

def reais(valor):
    """Formata um numero como moeda brasileira: 1234.5 -> 'R$ 1.234,50'."""
    texto = f'{valor:,.2f}'
    return 'R$ ' + texto.replace(',', 'X').replace('.', ',').replace('X', '.')

print('Conectado ao MySQL com sucesso.')

## Análise 1 – Os 5 órgãos com maior custo total
Realizado a soma total dos gastos com as viagens agrupado pelo órgão atrelado a viagem e ordenado de forma decrescente, para visualizar do custo maior ao menor.

O resultado está exibido em um gráfico de barras horizontal por facilitar a comparação entre categorias e acomodar nomes extensos dos órgãos.

In [ ]:
#Query SQL

SQL_ANALISE_1 = """
SELECT
    nome_orgao_superior,
    ROUND(SUM(valor_total),2) AS custo_total
FROM silver_viagem
GROUP BY nome_orgao_superior
ORDER BY custo_total DESC
LIMIT 5;
"""
dataframe_1 = consultar(SQL_ANALISE_1)
df1 = dataframe_1.copy()
df1["custo_total"] = df1["custo_total"].apply(reais)
display(df1)


#Plotagem do gráfico
plt.figure(figsize=(16,8))
plt.grid(axis='x', linestyle='--', alpha=0.4)

plt.barh(
    dataframe_1["nome_orgao_superior"],
    dataframe_1["custo_total"]
)

# Adiciona o valor ao final de cada barra
for i, valor in enumerate(dataframe_1["custo_total"]):
    plt.text(
        valor,                 # posição X
        i,                     # posição Y
        "  " + reais(valor),   # texto
        va='center',
        fontsize=10
    )

plt.title("TOP 5 Órgãos com maior custo total")
plt.xlabel("Custo total (R$)")
plt.ylabel("Órgão Superior")

plt.gca().invert_yaxis()

#Remove bordas dos eixos
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)


# Desativa a notação científica (1e8)
plt.gca().ticklabel_format(style='plain', axis='x')

# Exibe o eixo em milhões
plt.gca().xaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f'{x/1_000_000:.0f} mi')
)

plt.tight_layout()
plt.show()



## Análise 2 – Qual UF de destino aparece em mais trechos?
A consulta contabiliza a quantidade de trechos de viagem por unidade federativa (UF), permitindo identificar quais estados concentram o maior número de deslocamentos registrados.
A tabela `silver_itens`, ja tem o `subtotal`, então foi agrupado por `categoria` e utilizado `SUM(subtotal)`.

O gráfico de rosca representa intuitivamente a participação de cada UF no total de trechos registrados. Sua visualização facilita a percepção da proporção de cada estado em relação ao conjunto dos dados, tornando a análise mais objetiva.

In [ ]:
#Query SQL
SQL_ANALISE_2 = """
SELECT
    destino_uf,
    COUNT(*) AS quantidade_trechos
FROM silver_trecho
GROUP BY destino_uf
ORDER BY quantidade_trechos DESC
LIMIT 10;
"""

dataframe_2 = consultar(SQL_ANALISE_2)
display(dataframe_2)

#Plotagem do gráfico
plt.figure(figsize=(8,8))

plt.pie(
    dataframe_2["quantidade_trechos"],
    labels=dataframe_2["destino_uf"],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'width':0.45}
)

plt.title("Participação das UFs nos trechos de viagem")

plt.show()

## Análise 3 – Qual o meio de transporte mais utilizado nos trechos?

Esta consulta contabiliza a quantidade de trechos realizados por cada meio de transporte, identificando qual modalidade é utilizada com maior frequência nas viagens oficiais.

O gráfico de barras verticais foi utilizado para comparar a frequência entre diferentes meios de transporte, permitindo identificar facilmente qual modalidade é predominante.

In [ ]:
#Query SQL
SQL_ANALISE_3 = """
SELECT
    meio_transporte,
    COUNT(*) AS quantidade
FROM silver_trecho
GROUP BY meio_transporte
ORDER BY quantidade DESC
LIMIT 5;
"""

data_frame_3 = consultar(SQL_ANALISE_3)
display(data_frame_3)

plt.figure(figsize=(10,6))

plt.bar(
    data_frame_3["meio_transporte"],
    data_frame_3["quantidade"]
)

plt.title("Meios de transporte mais utilizados")
plt.xlabel("Meio de transporte")
plt.ylabel("Quantidade")

plt.xticks(rotation=15)

plt.tight_layout()
plt.show()

--
## Camada GOLD Agregada

**Tabela GOLD agregada:** Construção de uma camada GOLD com JOIN + GROUP BY com fucoes de
agregacao (`SUM`, `AVG`, `COUNT`, como tabela e como VIEW)


Cada registro representa uma viagem, contendo:

informações da viagem;
destino;
quantidade de trechos;
quantidade de pagamentos;
valor total pago;
valor médio dos pagamentos;
duração da viagem;
custo total da viagem.

Criada utilizando JOIN + GROUP BY e fica preparada para responder diversas perguntas de negócio.

| Pergunta                                                          | Tabela(s)                            | Tipo de gráfico    |                                                                 
| -----------------------------------------------------             | ------------------------------------ | ------------------ | 
| **Quais são os três destinos com maior custo médio por viagem?**  | `silver_viagem` + `silver_pagamento` | Barras horizontais | 
| **Qual UF de destino aparece em mais trechos?**                   | `silver_trecho`                      | Gráfico de Rosca   | 
| **Qual tipo de pagamento possui o maior valor médio?**            | `silver_trecho`                      | Barras             | 







In [ ]:
# TAREFA 1: Criar camada Gold com JOIN, como TABELA e como VIEW.

#==============================================================================
#   1. Órgãos Superiores (Responde: Os 5 órgãos com maior custo total)
#==============================================================================

SQL_AGREGADA_1 = """
DROP TABLE IF EXISTS gold_tb_custo_orgao_superior;

CREATE TABLE gold_tb_custo_orgao_superior AS
SELECT 
    v.nome_orgao_superior,
    COUNT(v.id_viagem) AS qtd_viagens,
    SUM(v.valor_total) AS custo_total,
    AVG(v.valor_total) AS custo_medio_por_viagem
FROM silver_viagem v
GROUP BY v.nome_orgao_superior;
"""

SQL_AGREGADA_VIEW_1 = """
DROP VIEW IF EXISTS vw_gold_custo_orgao_superior;

CREATE VIEW vw_gold_custo_orgao_superior AS 
SELECT * FROM gold_tb_custo_orgao_superior;

"""

# ==============================================================================
#   2. Destinos (Responde: Os 3 destinos com maior custo médio por viagem)
# ==============================================================================

SQL_AGREGADA_2 = """

DROP TABLE IF EXISTS gold_tb_custo_medio_destino;

CREATE TABLE gold_tb_custo_medio_destino AS
SELECT 
    t.destino_uf,
    t.destino_cidade,
    COUNT(DISTINCT v.id_viagem) AS qtd_viagens,
    AVG(v.valor_total) AS custo_medio_viagem,
    SUM(v.valor_total) AS custo_total_destino
FROM silver_trecho t
JOIN silver_viagem v ON t.id_viagem = v.id_viagem
WHERE t.destino_cidade IS NOT NULL
GROUP BY t.destino_uf, t.destino_cidade;
"""

SQL_AGREGADA_VIEW_2 = """

DROP VIEW IF EXISTS vw_gold_custo_medio_destino;

CREATE VIEW vw_gold_custo_medio_destino AS 
SELECT * FROM gold_tb_custo_medio_destino;

"""

# ==============================================================================
#   3. Viagens Longas (Responde: A viagem de maior duração e seu custo total)
# ==============================================================================

SQL_AGREGADA_3 = """

DROP TABLE IF EXISTS gold_tb_rank_viagens;

CREATE TABLE gold_tb_rank_viagens AS
SELECT 
    v.id_viagem,
    v.nome_viajante,
    v.destinos,
    v.duracao_dias,
    v.valor_total,
    SUM(p.valor) AS valor_pago_real,
    COUNT(t.id_trecho) AS qtd_trechos_percorridos
FROM silver_viagem v
LEFT JOIN silver_pagamento p ON v.id_viagem = p.id_viagem
LEFT JOIN silver_trecho t ON v.id_viagem = t.id_viagem
GROUP BY 
    v.id_viagem, 
    v.nome_viajante, 
    v.destinos, 
    v.duracao_dias, 
    v.valor_total;

"""

SQL_AGREGADA_VIEW_3 = """

DROP VIEW IF EXISTS vw_gold_rank_viagens;

CREATE VIEW vw_gold_rank_viagens AS 
SELECT * FROM gold_tb_rank_viagens;
"""

# ==============================================================================
#   4. Tipos de Pagamento (Responde: O tipo de pagamento com maior valor médio)
# ==============================================================================

SQL_AGREGADA_4 = """

DROP TABLE IF EXISTS gold_tb_tipo_pagamento;

CREATE TABLE gold_tb_tipo_pagamento AS
SELECT 
    p.tipo_pagamento,
    COUNT(p.id_pagamento) AS qtd_pagamentos,
    SUM(p.valor) AS valor_total,
    AVG(p.valor) AS valor_medio_pagamento
FROM silver_pagamento p
JOIN silver_viagem v ON p.id_viagem = v.id_viagem
GROUP BY p.tipo_pagamento;
"""
SQL_AGREGADA_VIEW_4 = """

DROP VIEW IF EXISTS vw_gold_tipo_pagamento;
CREATE VIEW vw_gold_tipo_pagamento AS 
SELECT * FROM gold_tb_tipo_pagamento;

"""

# ==============================================================================
# 5. Transporte (Responde: O meio de transporte mais usado nos trechos)
# ==============================================================================
SQL_AGREGADA_5 = """

DROP TABLE IF EXISTS gold_tb_meio_transporte;

CREATE TABLE gold_tb_meio_transporte AS
SELECT 
    t.meio_transporte,
    COUNT(t.id_trecho) AS qtd_trechos_utilizados,
    COUNT(DISTINCT t.id_viagem) AS qtd_viagens_atendidas
FROM silver_trecho t
JOIN silver_viagem v ON t.id_viagem = v.id_viagem
WHERE t.meio_transporte IS NOT NULL
GROUP BY t.meio_transporte;
"""

SQL_AGREGADA_VIEW_5 = """
DROP VIEW IF EXISTS vw_gold_meio_transporte;

CREATE VIEW vw_gold_meio_transporte AS 
SELECT * FROM gold_tb_meio_transporte;
"""


# ==============================================================================
#   6. Estados de Destino (Responde: A UF de destino que aparece em mais trechos)
# ==============================================================================
SQL_AGREGADA_6 = """
DROP VIEW IF EXISTS vw_gold_frequencia_uf;
DROP TABLE IF EXISTS gold_tb_frequencia_uf;

CREATE TABLE gold_tb_frequencia_uf AS
SELECT 
    t.destino_uf,
    COUNT(t.id_trecho) AS qtd_trechos_chegada,
    SUM(t.numero_diarias) AS total_diarias_uf
FROM silver_trecho t
JOIN silver_viagem v ON t.id_viagem = v.id_viagem
WHERE t.destino_uf IS NOT NULL
GROUP BY t.destino_uf;
"""
SQL_AGREGADA_VIEW_6 = """
CREATE VIEW vw_gold_frequencia_uf AS 
SELECT * FROM gold_tb_frequencia_uf;

"""

# ==============================================================================
#   7. Órgãos Pagadores (Responde: O órgão que pagou mais no total)
# ==============================================================================

SQL_AGREGADA_7 = """

DROP TABLE IF EXISTS gold_tb_orgao_pagador;

CREATE TABLE gold_tb_orgao_pagador AS
SELECT 
    p.nome_orgao_pagador,
    COUNT(p.id_pagamento) AS qtd_transacoes,
    SUM(p.valor) AS valor_total_pago
FROM silver_pagamento p
JOIN silver_viagem v ON p.id_viagem = v.id_viagem
WHERE p.nome_orgao_pagador IS NOT NULL
GROUP BY p.nome_orgao_pagador;
"""

SQL_AGREGADA_VIEW_7 = """
DROP VIEW IF EXISTS vw_gold_orgao_pagador;
CREATE VIEW vw_gold_orgao_pagador AS 
SELECT * FROM gold_tb_orgao_pagador;
"""


ANÁLISE 
2. Os 3 destinos com maior custo médio por viagem?
3. A viagem de maior duração e seu custo total?
4. Qual o tipo de pagamento com maior valor médio?
7. Qual órgão pagou mais no total?

In [ ]:
# Encerra a conexao com o banco
conexao.close()
print('Conexao encerrada.')